# Tests de mini-funcionalidades de clean_trips

Este notebook se usa para probar helpers y bloques internos de `clean_trips()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados

## Bloque 0. Preparación

### Imports generales

In [1]:
import json
from pathlib import Path

import h3
import numpy as np
import pandas as pd

### Imports del modulo

In [2]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.datasets import TripDataset

from pylondrina.transforms.cleaning import (
    CleanOptions,
    resolve_duplicates_subset_effective,
    mask_nulls_in_fields,
    mask_invalid_latlon,
    mask_invalid_h3,
    mask_origin_after_destination,
    mask_duplicates,
    mask_categorical_values,
    build_clean_summary,
    _normalize_string_list_or_none,
    _normalize_categorical_drop_map_or_none,
    _extract_temporal_tier,
    _extract_validated_flag,
    _is_valid_h3_value,
    _json_safe_scalar,
    _to_json_serializable_or_none,
)

### Helpers de apoyo para test

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def make_valid_h3(lat: float = -33.45, lon: float = -70.66, res: int = 8) -> str:
    return h3.latlng_to_cell(lat, lon, res)

### Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Bloque 1. Fixtures reutilizables mínimas

Qué se prueba en este bloque:
- estructuras mínimas para construir `TripDataset`
- schema simple reutilizable para varios helpers
- datasets pequeños y controlados

In [5]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
    )


def make_trip_dataset(
    data: pd.DataFrame,
    *,
    schema: TripSchema,
    metadata: dict | None = None,
    schema_effective: TripSchemaEffective | None = None,
) -> TripDataset:
    return TripDataset(
        data=data,
        schema=schema,
        schema_version=schema.version,
        provenance={"source": {"name": "synthetic_helper_test"}},
        field_correspondence={},
        value_correspondence={},
        metadata=metadata or {},
        schema_effective=schema_effective or TripSchemaEffective(),
    )

In [6]:
schema_min = make_trip_schema([
    make_field("movement_id", "string", required=True),
    make_field("user_id", "string", required=True),
    make_field("mode", "categorical", required=False, domain=DomainSpec(values=["walk", "bus", "metro", "unknown"])),
    make_field("purpose", "categorical", required=False, domain=DomainSpec(values=["work", "study", "unknown"])),
    make_field("origin_time_utc", "datetime", required=False),
    make_field("destination_time_utc", "datetime", required=False),
    make_field("origin_h3_index", "string", required=False),
    make_field("destination_h3_index", "string", required=False),
])

show_ok("Fixtures mínimas cargadas")

OK - Fixtures mínimas cargadas


## Bloque 2. Helpers principales de OP-04

### Test 2.1 - resolve_duplicates_subset_effective

Qué prueba:
- que devuelva `None` cuando la regla está inactiva
- que preserve orden y elimine duplicados en subset explícito
- que use el default `schema.required ∩ data.columns`
- que pueda devolver `[]` cuando el default queda vacío

In [13]:
df_dup = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "other": [1, 2],
})

trips_dup = make_trip_dataset(df_dup, schema=schema_min)

# Regla inactiva
assert resolve_duplicates_subset_effective(
    trips_dup,
    drop_duplicates=False,
    duplicates_subset=None,
) is None

# Subset explícito con duplicados y orden
subset_explicit = resolve_duplicates_subset_effective(
    trips_dup,
    drop_duplicates=True,
    duplicates_subset=["user_id", "movement_id", "user_id", "movement_id"],
)
assert subset_explicit == ["user_id", "movement_id"]

# Default desde schema.required intersectado con columnas presentes
subset_default = resolve_duplicates_subset_effective(
    trips_dup,
    drop_duplicates=True,
    duplicates_subset=None,
)
assert subset_default == ["movement_id", "user_id"]

# Default vacío
df_no_required = pd.DataFrame({"x": [1], "y": [2]})
trips_no_required = make_trip_dataset(df_no_required, schema=schema_min)
subset_empty = resolve_duplicates_subset_effective(
    trips_no_required,
    drop_duplicates=True,
    duplicates_subset=None,
)
assert subset_empty == []

show_ok("Test 2.1 - resolve_duplicates_subset_effective")

OK - Test 2.1 - resolve_duplicates_subset_effective


### Test 2.2 - mask_nulls_in_fields

Qué prueba:
- que marque filas con nulos en cualquiera de los campos objetivo
- que con lista vacía retorne todo False

In [14]:
df_nulls = pd.DataFrame({
    "a": [1, None, 3, 4],
    "b": ["x", "y", None, "z"],
    "c": [10, 20, 30, 40],
})

mask_ab = mask_nulls_in_fields(df_nulls, ["a", "b"])
assert mask_ab.tolist() == [False, True, True, False]

mask_empty = mask_nulls_in_fields(df_nulls, [])
assert mask_empty.tolist() == [False, False, False, False]

show_ok("Test 2.2 - mask_nulls_in_fields")

OK - Test 2.2 - mask_nulls_in_fields


### Test 2.3 - mask_invalid_latlon

Qué prueba:
- caso válido completo
- OD parcial válido (un extremo completo y el otro ausente)
- endpoint roto (lat sin lon, o viceversa)
- coordenadas fuera de rango
- ambos endpoints ausentes

Este test es especialmente importante porque aquí está una de las semánticas más delicadas de OP-04.

In [24]:
df_latlon = pd.DataFrame({
    "origin_latitude":      [-33.45, -33.45, -33.45, -95.00, None],
    "origin_longitude":     [-70.66, -70.66, None,   -70.66, None],
    "destination_latitude": [-33.46, None,   -33.46, -33.46, None],
    "destination_longitude":[-70.67, None,   -70.67, -70.67, None],
}, index=["valid_full", "od_partial_valid", "origin_broken", "origin_out_of_range", "both_absent"])

mask_latlon = mask_invalid_latlon(df_latlon)
print(bool(mask_latlon["valid_full"]))
assert not bool(mask_latlon["valid_full"])
assert not mask_latlon["od_partial_valid"] 
assert mask_latlon["origin_broken"] 
assert mask_latlon["origin_out_of_range"]
assert mask_latlon["both_absent"] 

show_ok("Test 2.3 - mask_invalid_latlon")

False
OK - Test 2.3 - mask_invalid_latlon


### Test 2.4 - mask_invalid_h3

Qué prueba:
- ambos H3 válidos
- H3 faltante
- H3 vacío
- H3 inválido

Aquí queremos asegurar la semántica estricta de H3: no se acepta parcialidad.

In [28]:
valid_h3_a = make_valid_h3(-33.45, -70.66, 8)
valid_h3_b = make_valid_h3(-33.46, -70.67, 8)

df_h3 = pd.DataFrame({
    "origin_h3_index": [valid_h3_a, valid_h3_a, "", "not_a_real_h3"],
    "destination_h3_index": [valid_h3_b, None, valid_h3_b, valid_h3_b],
}, index=["valid_pair", "missing_dest", "empty_origin", "invalid_origin"])

mask_h3 = mask_invalid_h3(df_h3)

assert not mask_h3["valid_pair"] 
assert mask_h3["missing_dest"] 
assert mask_h3["empty_origin"] 
assert mask_h3["invalid_origin"] 

show_ok("Test 2.4 - mask_invalid_h3")

OK - Test 2.4 - mask_invalid_h3


### Test 2.5 - mask_origin_after_destination

Qué prueba:
- origen antes que destino
- origen después que destino
- tiempos iguales
- nulos o parseo inválido no deben marcarse aquí

Este helper asume que la regla ya fue declarada evaluable; por eso solo probamos la comparación base.

In [29]:
df_time = pd.DataFrame({
    "origin_time_utc": [
        "2026-01-01T08:00:00Z",
        "2026-01-01T10:00:00Z",
        "2026-01-01T09:00:00Z",
        None,
        "not_a_datetime",
    ],
    "destination_time_utc": [
        "2026-01-01T09:00:00Z",
        "2026-01-01T09:30:00Z",
        "2026-01-01T09:00:00Z",
        "2026-01-01T09:00:00Z",
        "2026-01-01T10:00:00Z",
    ],
}, index=["valid_order", "reversed", "equal_times", "origin_null", "origin_invalid"])

mask_time = mask_origin_after_destination(df_time)

assert not mask_time["valid_order"] 
assert mask_time["reversed"] 
assert not mask_time["equal_times"]
assert not mask_time["origin_null"] 
assert not mask_time["origin_invalid"] 

show_ok("Test 2.5 - mask_origin_after_destination")

OK - Test 2.5 - mask_origin_after_destination


### Test 2.6 - mask_duplicates

Qué prueba:
- que se conserve la primera ocurrencia
- que solo se marquen duplicados posteriores

In [30]:
df_dups = pd.DataFrame({
    "user_id": ["u1", "u1", "u2", "u2", "u2"],
    "origin_h3_index": ["a", "a", "b", "b", "c"],
    "destination_h3_index": ["x", "x", "y", "y", "z"],
})

mask_dups = mask_duplicates(
    df_dups,
    subset=["user_id", "origin_h3_index", "destination_h3_index"],
)

assert mask_dups.tolist() == [False, True, False, True, False]

show_ok("Test 2.6 - mask_duplicates")

OK - Test 2.6 - mask_duplicates


### Test 2.7 - mask_categorical_values

Qué prueba:
- drop por valor categórico explícito
- sentinel `None` para drop de nulos
- combinación OR entre múltiples campos

In [32]:
df_cat = pd.DataFrame({
    "mode": ["walk", "unknown", "bus", None, "metro"],
    "purpose": ["work", "study", None, "unknown", "work"],
}, index=["r0", "r1", "r2", "r3", "r4"])

mask_cat = mask_categorical_values(
    df_cat,
    {
        "mode": ["unknown"],
        "purpose": ["unknown", None],
    },
)

assert not mask_cat["r0"]
assert mask_cat["r1"] 
assert mask_cat["r2"] 
assert mask_cat["r3"] 
assert not mask_cat["r4"]

show_ok("Test 2.7 - mask_categorical_values")

OK - Test 2.7 - mask_categorical_values


### Test 2.8 - build_clean_summary

Qué prueba:
- naming canónico del summary
- cálculo correcto de `dropped_total`
- relleno con cero de reglas no informadas
- serialización JSON-safe

In [33]:
summary = build_clean_summary(
    rows_in=10,
    rows_out=7,
    dropped_by_rule={
        "duplicates": 2,
        "invalid_h3": 1,
    },
)

assert summary["rows_in"] == 10
assert summary["rows_out"] == 7
assert summary["dropped_total"] == 3

expected_rule_keys = {
    "nulls_required",
    "nulls_fields",
    "invalid_latlon",
    "invalid_h3",
    "origin_after_destination",
    "duplicates",
    "categorical_values",
}
assert set(summary["dropped_by_rule"].keys()) == expected_rule_keys
assert summary["dropped_by_rule"]["duplicates"] == 2
assert summary["dropped_by_rule"]["invalid_h3"] == 1
assert summary["dropped_by_rule"]["nulls_required"] == 0

assert_json_safe(summary, "clean_summary")

show_ok("Test 2.8 - build_clean_summary")

OK - Test 2.8 - build_clean_summary


## Bloque 3. Helpers internos de uso general que sí conviene probar

### Test 3.1 - _normalize_string_list_or_none

Qué prueba:
- `None` -> `None`
- lista vacía -> `None`
- deduplicación preservando orden
- coerción simple a string

In [7]:
assert _normalize_string_list_or_none(None) is None
assert _normalize_string_list_or_none([]) is None
assert _normalize_string_list_or_none(["mode", "purpose", "mode"]) == ["mode", "purpose"]
assert _normalize_string_list_or_none(["a", 1, "a"]) == ["a", "1"]

show_ok("Test 3.1 - _normalize_string_list_or_none")

OK - Test 3.1 - _normalize_string_list_or_none


### Test 3.2 - _normalize_categorical_drop_map_or_none

Qué prueba:
- `None` y `{}` -> `None`
- deduplicación preservando orden por campo
- normalización JSON-safe
- preservación del sentinel `None`

In [8]:
assert _normalize_categorical_drop_map_or_none(None) is None
assert _normalize_categorical_drop_map_or_none({}) is None

normalized_drop_map = _normalize_categorical_drop_map_or_none({
    "mode": ["unknown", "unknown", None, None, "other"],
    "purpose": [None, "unknown", "unknown"],
})

assert normalized_drop_map == {
    "mode": ["unknown", None, "other"],
    "purpose": [None, "unknown"],
}

assert_json_safe(normalized_drop_map, "normalized_drop_map")

show_ok("Test 3.2 - _normalize_categorical_drop_map_or_none")

OK - Test 3.2 - _normalize_categorical_drop_map_or_none


### Test 3.3 - _extract_temporal_tier

Qué prueba:
- prioridad de metadata cuando ya trae tier válido
- inferencia conservadora de tier_1
- inferencia conservadora de tier_2
- fallback a tier_3

In [9]:
df_t1 = pd.DataFrame({
    "origin_time_utc": ["2026-01-01T08:00:00Z"],
    "destination_time_utc": ["2026-01-01T09:00:00Z"],
})

df_t2 = pd.DataFrame({
    "origin_time_local_hhmm": ["08:00"],
    "destination_time_local_hhmm": ["09:00"],
})

df_t3 = pd.DataFrame({
    "movement_id": ["m1"],
})

assert _extract_temporal_tier({"tier": "tier_2"}, df_t1) == "tier_2"
assert _extract_temporal_tier({}, df_t1) == "tier_1"
assert _extract_temporal_tier({}, df_t2) == "tier_2"
assert _extract_temporal_tier({}, df_t3) == "tier_3"

show_ok("Test 3.3 - _extract_temporal_tier")

OK - Test 3.3 - _extract_temporal_tier


### Test 3.4 - _extract_validated_flag

Qué prueba:
- lectura del flag top-level actual
- fallback legacy desde `metadata["flags"]["validated"]`
- comportamiento con metadata ausente o inválida

In [10]:
assert _extract_validated_flag({"is_validated": True}) is True
assert _extract_validated_flag({"is_validated": False}) is False
assert _extract_validated_flag({"flags": {"validated": True}}) is True
assert _extract_validated_flag({"flags": {"validated": False}}) is False
assert _extract_validated_flag({}) is False
assert _extract_validated_flag(None) is False

show_ok("Test 3.4 - _extract_validated_flag")

OK - Test 3.4 - _extract_validated_flag


### Test 3.5 - _is_valid_h3_value

Qué prueba:
- H3 válido
- `None`
- string vacío
- string inválido

In [11]:
valid_h3 = make_valid_h3(-33.45, -70.66, 8)

assert _is_valid_h3_value(valid_h3) is True
assert _is_valid_h3_value(None) is False
assert _is_valid_h3_value("") is False
assert _is_valid_h3_value("not_a_real_h3") is False

show_ok("Test 3.5 - _is_valid_h3_value")

OK - Test 3.5 - _is_valid_h3_value


### Test 3.6 - _json_safe_scalar y _to_json_serializable_or_none

Qué prueba:
- normalización básica de escalares problemáticos
- timestamps a ISO
- estructuras anidadas JSON-safe
- preservación razonable de tipos simples

In [12]:
ts = pd.Timestamp("2026-01-01T08:00:00Z")

assert _json_safe_scalar(None) is None
assert _json_safe_scalar("x") == "x"
assert _json_safe_scalar(5) == 5
print(_json_safe_scalar(np.nan))
assert _json_safe_scalar(np.nan) is None
assert _json_safe_scalar(ts).startswith("2026-01-01T08:00:00")

obj = {
    "a": 1,
    "b": [ts, np.nan, {"x": "y"}],
    "c": None,
}

serializable = _to_json_serializable_or_none(obj)
assert serializable["a"] == 1
assert serializable["b"][1] is None
assert isinstance(serializable["b"][0], str)

assert_json_safe(serializable, "serializable_nested_obj")

show_ok("Test 3.6 - _json_safe_scalar y _to_json_serializable_or_none")

None
OK - Test 3.6 - _json_safe_scalar y _to_json_serializable_or_none


## Bloque 4: Smoke tests integrados de clean_trips

### Grupo 4.1 - Fixtures mínimas reutilizables

Qué se prueba en este bloque:
- setup mínimo para usar la función pública `clean_trips`
- schema y metadata pequeñas, pero suficientes para atravesar el pipeline real
- datasets dummy reutilizables para smoke tests integrados

In [38]:
from copy import deepcopy

import pandas as pd

from pylondrina.reports import OperationReport
from pylondrina.transforms.cleaning import CleanOptions, clean_trips


def make_clean_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


BASE_CLEAN_SCHEMA = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_clean_field("movement_id", "string", required=True),
        "user_id": make_clean_field("user_id", "string", required=True),

        "origin_latitude": make_clean_field("origin_latitude", "float", required=False),
        "origin_longitude": make_clean_field("origin_longitude", "float", required=False),
        "destination_latitude": make_clean_field("destination_latitude", "float", required=False),
        "destination_longitude": make_clean_field("destination_longitude", "float", required=False),

        "origin_h3_index": make_clean_field("origin_h3_index", "string", required=False),
        "destination_h3_index": make_clean_field("destination_h3_index", "string", required=False),

        "origin_time_utc": make_clean_field("origin_time_utc", "datetime", required=False),
        "destination_time_utc": make_clean_field("destination_time_utc", "datetime", required=False),

        "mode": make_clean_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=["walk", "bus", "metro", "unknown"], extendable=False),
        ),
        "purpose": make_clean_field(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=["work", "study", "unknown"], extendable=False),
        ),
    },
    required=["movement_id", "user_id"],
    semantic_rules=None,
)


BASE_CLEAN_SCHEMA_EFFECTIVE = TripSchemaEffective(
    domains_effective={
        "mode": ["walk", "bus", "metro", "unknown"],
        "purpose": ["work", "study", "unknown"],
    },
    temporal={"tier": "tier_1"},
    fields_effective=list(BASE_CLEAN_SCHEMA.fields.keys()),
)


def make_tripdataset_for_clean(
    df: pd.DataFrame,
    *,
    schema: TripSchema = BASE_CLEAN_SCHEMA,
    schema_effective: TripSchemaEffective = BASE_CLEAN_SCHEMA_EFFECTIVE,
    is_validated: bool = True,
    temporal_tier: str = "tier_1",
    metadata: dict | None = None,
) -> TripDataset:
    base_metadata = {
        "dataset_id": "ds_clean_smoke",
        "events": [],
        "is_validated": is_validated,
        "domains_effective": {
            "mode": ["walk", "bus", "metro", "unknown"],
            "purpose": ["work", "study", "unknown"],
        },
        "temporal": {
            "tier": temporal_tier,
            "fields_present": list(df.columns),
        },
    }

    if metadata is not None:
        base_metadata.update(metadata)

    return TripDataset(
        data=df.copy(),
        schema=schema,
        schema_version=schema.version,
        provenance={"source": {"name": "synthetic_clean_smoke"}},
        field_correspondence={},
        value_correspondence={},
        metadata=base_metadata,
        schema_effective=schema_effective,
    )


show_ok("Grupo 4.1 - Fixtures mínimas reutilizables")

OK - Grupo 4.1 - Fixtures mínimas reutilizables


### Grupo 4.2 - happy path mínimo integrado con drops efectivos

Qué prueba:
- integración básica de la función pública en un caso correcto
- drops efectivos por reglas reales
- consistencia de `summary`, `parameters`, evento e `issues_summary`
- preservación de `is_validated`
- no mutación del dataset original

In [43]:
valid_h3_a = make_valid_h3(-33.45, -70.66, 8)
valid_h3_b = make_valid_h3(-33.46, -70.67, 8)
valid_h3_c = make_valid_h3(-33.47, -70.68, 8)
valid_h3_d = make_valid_h3(-33.48, -70.69, 8)

df_ok = pd.DataFrame({
    "movement_id": ["m1", "m2", "m3", "m4"],
    "user_id": ["u1", "u1", "u3", "u4"],
    "origin_latitude": [-33.45, -33.45, -33.47, -33.48],
    "origin_longitude": [-70.66, -70.66, -70.68, -70.69],
    "destination_latitude": [-33.46, -33.46, -33.48, -33.49],
    "destination_longitude": [-70.67, -70.67, -70.69, -70.70],
    "origin_h3_index": [valid_h3_a, valid_h3_a, valid_h3_c, valid_h3_d],
    "destination_h3_index": [valid_h3_b, valid_h3_b, valid_h3_d, valid_h3_a],
    "origin_time_utc": [
        "2026-04-01T08:00:00Z",
        "2026-04-01T08:00:00Z",   # duplicada respecto de m1 en subset elegido
        "2026-04-01T09:00:00Z",
        "2026-04-01T10:00:00Z",
    ],
    "destination_time_utc": [
        "2026-04-01T08:20:00Z",
        "2026-04-01T08:20:00Z",
        "2026-04-01T09:25:00Z",
        "2026-04-01T10:30:00Z",
    ],
    "mode": ["walk", "walk", "unknown", "bus"],
    "purpose": ["work", "work", "study", "study"],
})

trips = make_tripdataset_for_clean(df_ok, is_validated=True)
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_duplicates=True,
        duplicates_subset=["user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"],
        drop_rows_by_categorical_values={"mode": ["unknown"]},
    ),
)

assert isinstance(cleaned, TripDataset)
assert isinstance(report, OperationReport)
assert report.ok is True

assert report.summary["rows_in"] == 4
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 2
assert report.summary["dropped_by_rule"]["duplicates"] == 1
assert report.summary["dropped_by_rule"]["categorical_values"] == 1

assert report.parameters["drop_duplicates"] is True
assert report.parameters["duplicates_subset"] == ["user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"]
assert report.parameters["duplicates_subset_effective"] == ["user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"]

assert cleaned.metadata["is_validated"] is True
assert trips.metadata["events"] == events_before
assert len(cleaned.metadata["events"]) == 1

event = cleaned.metadata["events"][-1]
assert event["op"] == "clean_trips"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

assert list(cleaned.data["movement_id"]) == ["m1", "m4"]

pd.testing.assert_frame_equal(trips.data, data_before)

display(trips.data)
display(cleaned.data)
display(report.summary)
display(report.parameters)
show_ok("Grupo 4.2 - happy path mínimo integrado con drops efectivos")

,movement_id,user_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose
0,m1,u1,-33.45,-70.66,-33.46,-70.67,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,walk,work
1,m2,u1,-33.45,-70.66,-33.46,-70.67,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,walk,work
2,m3,u3,-33.47,-70.68,-33.48,-70.69,88b2c555d1fffff,88b2c5558bfffff,2026-04-01T09:00:00Z,2026-04-01T09:25:00Z,unknown,study
3,m4,u4,-33.48,-70.69,-33.49,-70.70,88b2c5558bfffff,88b2c55417fffff,2026-04-01T10:00:00Z,2026-04-01T10:30:00Z,bus,study


,movement_id,user_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose
0,m1,u1,-33.45,-70.66,-33.46,-70.67,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,walk,work
3,m4,u4,-33.48,-70.69,-33.49,-70.70,88b2c5558bfffff,88b2c55417fffff,2026-04-01T10:00:00Z,2026-04-01T10:30:00Z,bus,study


{'rows_in': 4,
 'rows_out': 2,
 'dropped_total': 2,
 'dropped_by_rule': {'nulls_required': 0,
  'nulls_fields': 0,
  'invalid_latlon': 0,
  'invalid_h3': 0,
  'origin_after_destination': 0,
  'duplicates': 1,
  'categorical_values': 1}}

{'drop_rows_with_nulls_in_required_fields': False,
 'drop_rows_with_nulls_in_fields': None,
 'drop_rows_with_invalid_latlon': False,
 'drop_rows_with_invalid_h3': False,
 'drop_rows_with_origin_after_destination': False,
 'drop_duplicates': True,
 'duplicates_subset': ['user_id',
  'origin_time_utc',
  'origin_h3_index',
  'destination_h3_index'],
 'duplicates_subset_effective': ['user_id',
  'origin_time_utc',
  'origin_h3_index',
  'destination_h3_index'],
 'drop_rows_by_categorical_values': {'mode': ['unknown']}}

OK - Grupo 4.2 - happy path mínimo integrado con drops efectivos


### Grupo 4.3 - sin cambios efectivos con regla activa

Qué prueba:
- la operación retorna normalmente aunque no elimine filas
- se emite el code de “sin cambios efectivos”
- se preserva `is_validated`
- se registra evento igualmente

In [41]:
df_no_changes = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "origin_h3_index": [valid_h3_a, valid_h3_c],
    "destination_h3_index": [valid_h3_b, valid_h3_d],
})

trips = make_tripdataset_for_clean(df_no_changes, is_validated=False)

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_with_invalid_h3=True,
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert report.summary["rows_in"] == 2
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 0
assert report.summary["dropped_by_rule"]["invalid_h3"] == 0

assert "CLN.NO_CHANGES.NO_ROWS_DROPPED" in codes

assert cleaned.metadata["is_validated"] is False
assert len(cleaned.metadata["events"]) == 1
assert cleaned.metadata["events"][-1]["op"] == "clean_trips"

show_ok("Grupo 4.3 - sin cambios efectivos con regla activa")

OK - Grupo 4.3 - sin cambios efectivos con regla activa


### Grupo 4.4 - warning degradado por regla temporal no evaluable

Qué prueba:
- política clave de OP-04: la regla temporal solo evalúa Tier 1
- en Tier 2 la operación no aborta
- se emite warning recuperable y retorna sin cambios efectivos

In [44]:
df_tier2 = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "origin_time_local_hhmm": ["08:30", "09:00"],
    "destination_time_local_hhmm": ["08:10", "09:20"],
    "mode": ["walk", "bus"],
})

trips = make_tripdataset_for_clean(
    df_tier2,
    is_validated=True,
    temporal_tier="tier_2",
)

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_with_origin_after_destination=True,
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "CLN.RULE.TEMPORAL_RULE_NOT_EVALUABLE" in codes
assert "CLN.NO_CHANGES.NO_RULES_ACTIVE" in codes

assert report.summary["rows_in"] == 2
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 0
assert report.summary["dropped_by_rule"]["origin_after_destination"] == 0

assert cleaned.metadata["is_validated"] is True
assert len(cleaned.metadata["events"]) == 1
assert cleaned.metadata["events"][-1]["op"] == "clean_trips"

show_ok("Grupo 4.4 - warning degradado por regla temporal no evaluable")

OK - Grupo 4.4 - warning degradado por regla temporal no evaluable


### Grupo 4.5 - política clave de duplicados con subset por default

Qué prueba:
- `duplicates_subset=None` con `drop_duplicates=True`
- uso del default `schema.required ∩ data.columns`
- persistencia de `duplicates_subset_effective` en report y evento

In [45]:
schema_dup_default = TripSchema(
    version="1.1",
    fields={
        "user_id": make_clean_field("user_id", "string", required=True),
        "origin_time_utc": make_clean_field("origin_time_utc", "datetime", required=True),
        "mode": make_clean_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=["walk", "bus", "unknown"], extendable=False),
        ),
    },
    required=["user_id", "origin_time_utc"],
    semantic_rules=None,
)

df_dup_default = pd.DataFrame({
    "user_id": ["u1", "u1", "u2"],
    "origin_time_utc": [
        "2026-04-01T08:00:00Z",
        "2026-04-01T08:00:00Z",
        "2026-04-01T09:00:00Z",
    ],
    "mode": ["walk", "walk", "bus"],
})

trips = make_tripdataset_for_clean(
    df_dup_default,
    schema=schema_dup_default,
    is_validated=True,
)

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_duplicates=True,
        duplicates_subset=None,
    ),
)

assert report.ok is True
assert report.parameters["duplicates_subset"] is None
assert report.parameters["duplicates_subset_effective"] == ["user_id", "origin_time_utc"]

assert report.summary["rows_in"] == 3
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 1
assert report.summary["dropped_by_rule"]["duplicates"] == 1

event = cleaned.metadata["events"][-1]
assert event["parameters"]["duplicates_subset"] is None
assert event["parameters"]["duplicates_subset_effective"] == ["user_id", "origin_time_utc"]

show_ok("Grupo 4.5 - política clave de duplicados con subset por default")

OK - Grupo 4.5 - política clave de duplicados con subset por default


### Grupo 4.6 - fatal de configuración sin evento

Qué prueba:
- aborto fatal por subset explícito imposible
- no se registra evento en el dataset de entrada
- no se altera `is_validated`
- no se muta el dataframe original

In [46]:
df_fatal = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "mode": ["walk", "bus"],
})

trips = make_tripdataset_for_clean(df_fatal, is_validated=True)
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    clean_trips(
        trips,
        options=CleanOptions(
            drop_duplicates=True,
            duplicates_subset=["missing_field"],
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValueError)
assert getattr(raised, "code", None) == "CLN.CONFIG.INVALID_DUPLICATES_SUBSET"

assert trips.metadata["events"] == events_before
assert trips.metadata["is_validated"] == validated_before
pd.testing.assert_frame_equal(trips.data, data_before)

show_ok("Grupo 4.6 - fatal de configuración sin evento")

OK - Grupo 4.6 - fatal de configuración sin evento
